## 1. Load the original dataset

In [18]:
import numpy as np
import pandas as pd

df = pd.read_csv("../data/raw/student-mat.csv", sep=";")

## 2. Handle missing values

First, we count the missing values for each column.

In [19]:
df.isnull().sum()

school         7
sex           17
age            2
address       16
famsize       11
Pstatus       18
Medu          12
Fedu           6
Mjob           2
Fjob          18
reason         9
guardian       8
traveltime    17
studytime     12
failures      17
schoolsup     13
famsup        16
paid          15
activities     2
nursery        2
higher        20
internet       8
romantic      12
famrel         7
freetime      18
goout          7
Dalc          12
Walc           2
health         6
absences       3
G1             3
G2             3
G3             9
dtype: int64

Then we replace the numeric missing values with the median of thier columns and for the categorical values we will use the mode.

In [20]:
num_cols = df.select_dtypes(include=["number"])
cat_cols = df.select_dtypes(include=["object", "category"]).columns
#Numeric
for col in num_cols :
    df[col] = df[col].fillna(df[col].median())
#Categoricaldf
for col in cat_cols :
    df[col] = df[col].fillna(df[col].mode()[0])


C:\Users\user\AppData\Local\Temp\ipykernel_12664\65528414.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=["object", "category"]).columns


In [21]:
df.isnull().sum()

school        0
sex           0
age           0
address       0
famsize       0
Pstatus       0
Medu          0
Fedu          0
Mjob          0
Fjob          0
reason        0
guardian      0
traveltime    0
studytime     0
failures      0
schoolsup     0
famsup        0
paid          0
activities    0
nursery       0
higher        0
internet      0
romantic      0
famrel        0
freetime      0
goout         0
Dalc          0
Walc          0
health        0
absences      0
G1            0
G2            0
G3            0
dtype: int64

## 3. Handle duplicates

we count the number of duplicates.

In [22]:
df.duplicated().sum()

np.int64(20)

we drope the duplicates.

In [23]:
df = df.drop_duplicates()

## 4. Handle outliers

we define the outliers with the IQR and we handle them with the capping method.

In [24]:
# num_cols already defined ,refers to numric columns 
for col in num_cols :
    Q1 , Q3 = df[col].quantile([0.25,0.75])
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df[col].clip(lower=lower_bound , upper=upper_bound)

## 5. Encode categorical variables

First identify the categorical features:

In [25]:
categorical_cols = df.select_dtypes(
    include="object"
).columns

C:\Users\user\AppData\Local\Temp\ipykernel_12664\884297372.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(


For nominal and non binary variables :

In [26]:
# we will use OneHotEncoder for the non binary and nominal variables 
from sklearn.preprocessing import OneHotEncoder

multi_cols = ["Mjob", "Fjob", "reason", "guardian"]

ohe = OneHotEncoder(sparse_output=False,handle_unknown='ignore')

encoded = ohe.fit_transform(df[multi_cols])
#delete the original columns 

df = df.drop(columns=multi_cols)

#encoded dataframe construction
df_encoded = pd.DataFrame(
    encoded ,
    columns = ohe.get_feature_names_out(),
    index=df.index
)

#concatenate the encoded datframe to the original one 

df = pd.concat([df , df_encoded] , axis=1)


For ordinal and  binary variables :

In [27]:
# and LabelEncoder for binary and ordinal variables
from sklearn.preprocessing import LabelEncoder

binary_cols = [
    "school", "sex", "address", "famsize", "Pstatus",
    "schoolsup", "famsup", "paid", "activities",
    "nursery", "higher", "internet", "romantic",
]
#define the model 
le = LabelEncoder()
#apply the transformation for each column
for col in binary_cols :
    df[col] = le.fit_transform(df[col])


## 6. Feature scaling 

For numerical features:

In [28]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
#all the variables are already numericals
#so we apply the satandard scaler ( mean = 0 and std = 1)
df = pd.DataFrame(scaler.fit_transform(df) , columns=df.columns)

## 7. Save the processed dataset

save the processed data to "student_performance_cleaned.csv"

In [29]:
df.to_csv(
    "../data/processed/student_performance_cleaned.csv",
    index=False
)